# Stage 3 Qwen2.5-VL-3B LoRA SFT Repair Clean

Clean supervised adaptation smoke for the Stage 3 reporter. The run keeps the same clean train/val split, the same `vlm_labels_v1` fields, and the no-crop-path prompt policy.

The notebook is intentionally gated: it trains a small LoRA adapter, checks whether the model can reproduce valid JSON and correct coarse labels on a few training examples, and only then runs full clean `val_v2` evaluation.


In [ ]:
from pathlib import Path
import json
import os
import random
import re
import shutil
import subprocess
from collections import Counter

REPO_URL = "https://github.com/konstRyaz/vlm-for-insulator-defect-detection.git"
REPO_DIR = Path("/kaggle/working/vlm-for-insulator-defect-detection")
DATASET_ROOT_CANDIDATES = [
    Path("/kaggle/input/datasets/kostyaryazanov/idid-coco-v3"),
    Path("/kaggle/input/idid-coco-v3"),
]
STAGE3_REL = Path("stage3_regrouped_v2")
TRAIN_JSONL_REL = STAGE3_REL / "train_balanced/vlm_labels_v1_train_balanced_v2.annotated.jsonl"
VAL_JSONL_REL = STAGE3_REL / "val/vlm_labels_v1_val_v2.annotated.jsonl"

MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"
RUN_NAME = "stage3_qwen25vl_3b_lora_sft_repair_clean"
PROMPT_VERSION = "qwen_vlm_labels_v1_prompt_v7f_flashover_unclear_to_unknown_nocroppath"

MAX_TRAIN_SAMPLES = 96
OVERFIT_CHECK_SAMPLES = 6
MIN_OVERFIT_PARSE_OK = 6
MIN_OVERFIT_COARSE_OK = 4
NUM_EPOCHS = 1
LR = 5e-5
MAX_NEW_TOKENS = 220
MAX_PIXELS = 401408
SEED = 42

print("RUN_NAME:", RUN_NAME)
print("MODEL_ID:", MODEL_ID)


In [ ]:
def sh(args, cwd: Path | None = None, check: bool = True):
    print("$", " ".join(str(a) for a in args))
    p = subprocess.run([str(a) for a in args], cwd=str(cwd) if cwd else None, text=True, capture_output=True)
    if p.stdout:
        print(p.stdout)
    if p.stderr:
        print(p.stderr)
    if check and p.returncode != 0:
        raise RuntimeError(f"Command failed ({p.returncode})")
    return p

def load_jsonl(path: Path):
    rows = []
    with path.open(encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return rows

def write_jsonl(path: Path, rows):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

def resolve_crop(row, split_root: Path) -> Path:
    p = Path(row["crop_path"])
    if p.is_absolute() and p.exists():
        return p
    candidates = [split_root / p, split_root.parent / p]
    for cand in candidates:
        if cand.exists():
            return cand
    raise FileNotFoundError(row["crop_path"])

def stratified_sample(rows, label_key="coarse_class", max_total=96, seed=42):
    rng = random.Random(seed)
    buckets = {}
    for row in rows:
        buckets.setdefault(row.get(label_key, "unknown"), []).append(row)
    for bucket in buckets.values():
        rng.shuffle(bucket)
    labels = sorted(buckets)
    out = []
    while len(out) < max_total and any(buckets.values()):
        for label in labels:
            if buckets[label] and len(out) < max_total:
                out.append(buckets[label].pop())
    rng.shuffle(out)
    return out


In [ ]:
gpu = sh(["nvidia-smi"], check=False)
if gpu.returncode != 0:
    raise RuntimeError("GPU is required for this run")


In [ ]:
DATA_ROOT = None
for root in DATASET_ROOT_CANDIDATES:
    if (root / TRAIN_JSONL_REL).exists() and (root / VAL_JSONL_REL).exists():
        DATA_ROOT = root
        break
if DATA_ROOT is None:
    raise FileNotFoundError("Could not find clean Stage 3 train/val JSONL")

train_jsonl = DATA_ROOT / TRAIN_JSONL_REL
val_jsonl = DATA_ROOT / VAL_JSONL_REL
train_root = train_jsonl.parent
val_root = val_jsonl.parent
print("DATA_ROOT:", DATA_ROOT)
print("train_jsonl:", train_jsonl)
print("val_jsonl:", val_jsonl)


In [ ]:
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
sh(["git", "clone", REPO_URL, str(REPO_DIR)])
sh([
    "python", "-m", "pip", "install", "-q",
    "--extra-index-url", "https://download.pytorch.org/whl/cu121",
    "torch==2.5.1", "torchvision==0.20.1",
    "transformers==4.57.1", "accelerate", "qwen-vl-utils", "peft", "bitsandbytes", "pyyaml", "tabulate",
], cwd=REPO_DIR)
print("Repo ready:", REPO_DIR)


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from qwen_vl_utils import process_vision_info

random.seed(SEED)
torch.manual_seed(SEED)

system_prompt = (REPO_DIR / "configs/pipeline/prompts/stage3_vlm_system_v7f_flashover_unclear_to_unknown.txt").read_text(encoding="utf-8")
user_template = (REPO_DIR / "configs/pipeline/prompts/stage3_vlm_user_v6d_balanced_notaglock_nocroppath.txt").read_text(encoding="utf-8")

FORBIDDEN_HINTS = ["crop_path", "defect_broken/", "defect_flashover/", "insulator_ok/"]
for hint in FORBIDDEN_HINTS:
    assert hint not in system_prompt
    assert hint not in user_template

ALLOWED_CLASSES = {"insulator_ok", "defect_flashover", "defect_broken"}

def render_user(row):
    text = user_template
    safe_meta = {
        "record_id": row.get("record_id", ""),
        "split": row.get("split", ""),
        "source": row.get("source", ""),
        "bbox_xywh": row.get("bbox_xywh", ""),
    }
    for key, value in safe_meta.items():
        text = text.replace("{{" + key + "}}", str(value))
    return text

def target_json(row):
    obj = {
        "coarse_class": row["coarse_class"],
        "visual_evidence_tags": row.get("visual_evidence_tags", []),
        "visibility": row.get("visibility", "clear"),
        "short_canonical_description_en": row.get("short_canonical_description_en") or row.get("short_canonical_description", ""),
        "report_snippet_en": row.get("report_snippet_en") or row.get("report_snippet", ""),
    }
    return json.dumps(obj, ensure_ascii=False, separators=(",", ":"))

all_train_rows = [r for r in load_jsonl(train_jsonl) if r.get("coarse_class") in ALLOWED_CLASSES]
train_rows = stratified_sample(all_train_rows, max_total=MAX_TRAIN_SAMPLES, seed=SEED)
val_rows = load_jsonl(val_jsonl)
print("train rows:", len(train_rows), Counter(r["coarse_class"] for r in train_rows))
print("val rows:", len(val_rows), Counter(r["coarse_class"] for r in val_rows))

processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True, max_pixels=MAX_PIXELS)
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type="nf4")
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb,
    device_map="auto",
    trust_remote_code=True,
    attn_implementation="eager",
)
model = prepare_model_for_kbit_training(model)
model.config.use_cache = False

lora_cfg = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()


In [ ]:
class VLDataset(Dataset):
    def __init__(self, rows, root):
        self.rows = rows
        self.root = root
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, idx):
        row = self.rows[idx]
        image_uri = resolve_crop(row, self.root).resolve().as_uri()
        messages = [
            {"role": "system", "content": [{"type": "text", "text": system_prompt}]},
            {"role": "user", "content": [{"type": "image", "image": image_uri}, {"type": "text", "text": render_user(row)}]},
        ]
        return {"row": row, "messages": messages, "target": target_json(row)}

def build_train_inputs(messages, target):
    # Use one prompt template and append the assistant JSON manually. This avoids the
    # chat-template mismatch that made the previous LoRA smoke unstable.
    prompt_text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    full_text = prompt_text + target + processor.tokenizer.eos_token
    images, videos = process_vision_info(messages)
    prompt_inputs = processor(text=[prompt_text], images=images, videos=videos, padding=False, return_tensors="pt")
    full_inputs = processor(text=[full_text], images=images, videos=videos, padding=False, return_tensors="pt")
    labels = full_inputs["input_ids"].clone()
    prompt_len = min(prompt_inputs["input_ids"].shape[1], labels.shape[1])
    labels[:, :prompt_len] = -100
    labels[labels == processor.tokenizer.pad_token_id] = -100
    full_inputs["labels"] = labels
    return full_inputs

def collate(batch):
    item = batch[0]
    return build_train_inputs(item["messages"], item["target"])

loader = DataLoader(VLDataset(train_rows, train_root), batch_size=1, shuffle=True, collate_fn=collate)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
model.train()
losses = []
for epoch in range(NUM_EPOCHS):
    for step, batch in enumerate(loader, start=1):
        batch = {k: v.to(model.device) if hasattr(v, "to") else v for k, v in batch.items()}
        out = model(**batch)
        loss = out.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)
        losses.append(float(loss.detach().cpu()))
        if step % 16 == 0:
            print("epoch", epoch + 1, "step", step, "recent_loss", sum(losses[-16:]) / len(losses[-16:]))
print("training done", "mean_loss", sum(losses) / len(losses))


In [ ]:
adapter_dir = REPO_DIR / "outputs" / RUN_NAME / "adapter"
adapter_dir.mkdir(parents=True, exist_ok=True)
model.save_pretrained(adapter_dir)
processor.save_pretrained(adapter_dir)
print("adapter_dir:", adapter_dir)


In [ ]:
out_root = REPO_DIR / "outputs/stage3_vlm_baseline_runs" / RUN_NAME
out_root.mkdir(parents=True, exist_ok=True)

def extract_json(text):
    m = re.search(r"\{.*\}", text, flags=re.S)
    if not m:
        raise ValueError("no JSON object")
    return json.loads(m.group(0))

def normalize_prediction(obj):
    pred = {
        "coarse_class": obj.get("coarse_class", "unknown"),
        "visual_evidence_tags": obj.get("visual_evidence_tags", []),
        "visibility": obj.get("visibility", "ambiguous"),
        "short_canonical_description_en": obj.get("short_canonical_description_en", ""),
        "report_snippet_en": obj.get("report_snippet_en", ""),
    }
    if pred["coarse_class"] not in {"insulator_ok", "defect_flashover", "defect_broken", "unknown"}:
        pred["coarse_class"] = "unknown"
    if pred["visibility"] not in {"clear", "partial", "ambiguous"}:
        pred["visibility"] = "ambiguous"
    if not isinstance(pred["visual_evidence_tags"], list):
        pred["visual_evidence_tags"] = []
    return pred

def generate_for_rows(rows, root, output_prefix=None):
    sample_rows, raw_rows, parsed_rows, pred_rows = [], [], [], []
    model.eval()
    with torch.no_grad():
        for idx, row in enumerate(rows, start=1):
            image_uri = resolve_crop(row, root).resolve().as_uri()
            messages = [
                {"role": "system", "content": [{"type": "text", "text": system_prompt}]},
                {"role": "user", "content": [{"type": "image", "image": image_uri}, {"type": "text", "text": render_user(row)}]},
            ]
            text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            images, videos = process_vision_info(messages)
            inputs = processor(text=[text], images=images, videos=videos, padding=True, return_tensors="pt").to(model.device)
            gen = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
            gen_trim = gen[:, inputs.input_ids.shape[1]:]
            raw_text = processor.batch_decode(gen_trim, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]
            status = "ok"
            errors = []
            try:
                norm = normalize_prediction(extract_json(raw_text))
            except Exception as exc:
                status = "parse_error"
                errors.append(str(exc))
                norm = {
                    "coarse_class": "unknown",
                    "visual_evidence_tags": ["ambiguous_evidence"],
                    "visibility": "ambiguous",
                    "short_canonical_description_en": "Uncertain crop.",
                    "report_snippet_en": "The crop should be reviewed.",
                }
            base = {k: row.get(k) for k in ["record_id", "image_id", "box_id", "source", "split", "bbox_xywh", "crop_path", "image_path", "score", "category_name", "label_version"]}
            pred = dict(base)
            pred.update(norm)
            pred["needs_review"] = norm.get("visibility") == "ambiguous"
            pred["short_canonical_description"] = norm.get("short_canonical_description_en", "")
            pred["report_snippet"] = norm.get("report_snippet_en", "")
            sample_rows.append({"record_id": row["record_id"], "status": status, "backend_mode": "qwen_hf_lora_sft_repair", "model_id": MODEL_ID})
            raw_rows.append({"record_id": row["record_id"], "raw_text": raw_text})
            parsed_rows.append({"record_id": row["record_id"], "status": status, "normalization_errors": errors, "normalized_prediction": norm})
            pred_rows.append(pred)
            if idx % 10 == 0:
                print("generated", idx, "/", len(rows))
    if output_prefix:
        write_jsonl(out_root / f"{output_prefix}_sample_results.jsonl", sample_rows)
        write_jsonl(out_root / f"{output_prefix}_raw_responses.jsonl", raw_rows)
        write_jsonl(out_root / f"{output_prefix}_parsed_predictions.jsonl", parsed_rows)
        write_jsonl(out_root / f"{output_prefix}_predictions_vlm_labels_v1.jsonl", pred_rows)
    return sample_rows, raw_rows, parsed_rows, pred_rows

overfit_rows = train_rows[:OVERFIT_CHECK_SAMPLES]
overfit_sample, overfit_raw, overfit_parsed, overfit_pred = generate_for_rows(overfit_rows, train_root, output_prefix="overfit_check")
overfit_parse_ok = sum(1 for r in overfit_parsed if r["status"] == "ok")
overfit_coarse_ok = sum(1 for gt, pred in zip(overfit_rows, overfit_pred) if gt.get("coarse_class") == pred.get("coarse_class"))
print("overfit_parse_ok", overfit_parse_ok, "/", len(overfit_rows))
print("overfit_coarse_ok", overfit_coarse_ok, "/", len(overfit_rows))

if overfit_parse_ok < MIN_OVERFIT_PARSE_OK or overfit_coarse_ok < MIN_OVERFIT_COARSE_OK:
    print("Overfit gate failed; full validation skipped.")
    sample_rows, raw_rows, parsed_rows, pred_rows = overfit_sample, overfit_raw, overfit_parsed, overfit_pred
else:
    print("Overfit gate passed; running full clean val.")
    sample_rows, raw_rows, parsed_rows, pred_rows = generate_for_rows(val_rows, val_root)

write_jsonl(out_root / "sample_results.jsonl", sample_rows)
write_jsonl(out_root / "raw_responses.jsonl", raw_rows)
write_jsonl(out_root / "parsed_predictions.jsonl", parsed_rows)
write_jsonl(out_root / "predictions_vlm_labels_v1.jsonl", pred_rows)
print("wrote", out_root)


In [ ]:
validate_cmd = ["python", "scripts/validate_vlm_labels_v1.py", "--input", str(out_root / "predictions_vlm_labels_v1.jsonl")]
sh(validate_cmd, cwd=REPO_DIR, check=False)

eval_dir = out_root / "eval"
sh([
    "python", "scripts/eval_stage3_vlm_baseline.py",
    "--run-dir", str(out_root),
    "--ground-truth-jsonl", str(val_jsonl),
    "--output-dir", str(eval_dir),
], cwd=REPO_DIR, check=False)

summary = {
    "run_name": RUN_NAME,
    "model_id": MODEL_ID,
    "train_samples": len(train_rows),
    "overfit_check_samples": OVERFIT_CHECK_SAMPLES,
    "overfit_parse_ok": overfit_parse_ok,
    "overfit_coarse_ok": overfit_coarse_ok,
    "full_val_attempted": len(pred_rows) == len(val_rows),
    "output_dir": str(out_root),
}
(out_root / "lora_repair_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(json.dumps(summary, indent=2))


In [ ]:
package_root = REPO_DIR / "outputs" / "stage3_qwen_lora_sft_repair_package"
if package_root.exists():
    shutil.rmtree(package_root)
package_root.mkdir(parents=True, exist_ok=True)
shutil.copytree(out_root, package_root / RUN_NAME)
archive_path = Path("/kaggle/working/stage3_deliverables_qwen25vl_3b_lora_sft_repair_clean.tar.gz")
if archive_path.exists():
    archive_path.unlink()
sh(["tar", "-czf", str(archive_path), "-C", str(package_root.parent), package_root.name], check=True)
print("Artifact archive:", archive_path)
shutil.rmtree(REPO_DIR, ignore_errors=True)
print("Workspace cleaned")
